In [ ]:
from utils.utils import set_seed
import torch

SEED = 2952
set_seed(SEED, deterministic=True, benchmark=True) # benchmark=True for speed; False for reproducibility
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
arch = 'vit_s'
batch_size = 256
optimizer = "adamw"
lr = 1.5e-4
wd = .1
momentum = 0.9
epochs = 300
warmup_epochs = 40
stop_grad_conv1 = True
moco_m_cos = True
moco_t = .2
crop_min = .2
workers = 32
moco_dim = 256
moco_mlp_dim = 4096
moco_m = 0.99
print_freq = 10
lr = lr * batch_size / 256
ckpt_dir = f'./checkpoints_{arch}'

In [ ]:
from data.data import get_ssl_dataloader

train_loader = get_ssl_dataloader(batch_size=batch_size, workers=workers)

In [ ]:
from models.models import MoCo

model = MoCo(
    encoder_name=arch,
    dim=moco_dim,
    mlp_dim=moco_mlp_dim,
    T=moco_t,
    proj_layers=3 if 'vit' in arch else 2,
    pred_layers=2,
)

In [ ]:
from scripts.ssl_script import train_moco

train_moco(
    model=model,
    train_loader=train_loader,
    epochs=epochs,
    warmup_epochs=warmup_epochs,
    moco_m=moco_m,
    moco_m_cos=moco_m_cos,
    lr=lr,
    wd=wd,
    momentum=momentum,
    optimizer=optimizer,
    device=device,
    ckpt_dir=ckpt_dir,
    print_freq=10,
    save_freq=50,
)